# Problem 5: Multi-Node Jobs

---

> Place all files for this problem in a directory called `problem_5`.

## Overview

So far, all of your batch jobs have run on a single node. But SLURM can allocate **multiple nodes** to a single job, allowing you to distribute work across the cluster. In this problem, you will use a Shakespearean sentence generator to learn how multi-node execution works — and discover a common mistake that causes your job to silently ignore the extra nodes you requested.

The [Infinite Monkey Theorem](https://en.wikipedia.org/wiki/Infinite_monkey_theorem) proposes that a monkey hitting keys at random on a typewriter for an infinite amount of time will eventually produce the complete works of Shakespeare. We don't have infinite time or monkeys — but we do have a cluster. Let's see what 6 nodes can come up with.


## Part A: The Sentence Generator

Create a Python script called `shakespeare.py` that generates random Shakespearean-style sentences. The script should:

- Accept a command-line argument for the number of sentences to generate
- Print each sentence prefixed with the **hostname** of the machine it ran on
- Use `random.choice()` to assemble sentences from lists of subjects, verbs, objects, and adverbs

Here is a starter to get you going — feel free to expand or modify the word lists:

```python
import random
import socket

subjects = [
    "A noble king", "A wicked fool", "The fair maiden", "A dastardly villain",
    "Yon humble servant", "A lonely ghost", "The cackling witch", "A valiant knight",
    "This star-cross'd lover", "A secret heir", "The wise apothecary",
    "A banished prince", "The jealous moor", "A scheming duchess", "The melancholy Dane",
    "A wandering minstrel", "The vengeful spirit", "A pious friar", "The mad prophetess",
    "A treacherous courtier", "The weary soldier", "A cursed soothsayer",
    "The scorned enchantress", "A reformed rogue", "The rightful sovereign",
    "A shipwrecked mariner", "The blind oracle", "A disguised noblewoman",
    "The tormented jester", "A lovestruck shepherd", "The exiled sorcerer",
    "A doomed conspirator", "The aging monarch", "A repentant murderer",
    "The phantom of the tower", "A silver-tongued diplomat", "The woodland fairy",
    "A battle-scarred general", "The forsaken bride", "A cunning alchemist"
]

verbs = [
    "doth proclaim", "hath stolen", "shall embrace", "doth beseech", "hath forsaken",
    "doth conspire against", "shall vanquish", "hath witnessed", "doth seeketh",
    "shall betray", "doth professeth", "hath summoned", "shall lament",
    "doth unravel", "hath devoured", "shall renounce", "doth whispereth of",
    "hath concealed", "shall conjure", "doth mourn", "hath shattered",
    "shall inherit", "doth feign ignorance of", "hath unearthed",
    "shall set ablaze", "doth harbour", "hath squandered",
    "shall exorcise", "doth covet", "hath defied"
]

objects = [
    "a king's ransom", "a secret sonnet", "a most grievous fate", "the Queen's favour",
    "a poisoned chalice", "the midnight hour", "a ghostly apparition", "a hero's honour",
    "a lover's vow", "a cursed prophecy", "a pound of flesh",
    "the throne of England", "a blood-stained dagger", "the enchanted forest",
    "a forbidden manuscript", "the crown jewels", "a severed head upon a pike",
    "a tempest most foul", "the last of the wine", "a forged letter of passage",
    "the dark tower", "a chest of doubloons", "the philosopher's stone",
    "a map to buried treasure", "the keys to the dungeon", "a vial of tears",
    "an ancient grudge", "the deed to the castle", "a merchant's debt",
    "the bones of a saint"
]

adverbs = [
    "with great sorrow", "ere long", "with haste", "anon", "in jest", "perchance",
    "with villainous intent", "forsooth", "in truth", "under the pale moonlight",
    "whilst the court doth sleep", "upon the morrow", "with trembling hands",
    "before the cock doth crow", "behind the velvet curtain", "amid thunder and lightning",
    "in the dead of winter", "with a heavy heart", "by the light of a single candle",
    "as the plague approacheth", "without mercy or remorse", "to the sound of distant drums",
    "whilst disguised as a peasant", "at the stroke of midnight",
    "with a wink and a prayer", "on bended knee", "through gritted teeth",
    "in defiance of the gods", "with poison upon their lips", "as fate would have it"
]
```

Test it locally or in an `sinteractive` session:

```bash
python3 shakespeare.py 5
```

You should see 5 random sentences, each prefixed with the hostname.


## Part B: The One-Node Mistake

Write a batch script called `shakespeare_broken.sbatch` that requests **6 nodes** but runs the script **without** `srun`:

```bash
#!/bin/bash
#SBATCH --job-name=shakespeare
#SBATCH --account=mpcs56430
#SBATCH --partition=caslake
#SBATCH --nodes=6
#SBATCH --ntasks-per-node=1
#SBATCH --time=00:05:00
#SBATCH --output=shakespeare-%j.out
#SBATCH --error=shakespeare-%j.err

echo "Writing our great work on $SLURM_JOB_NUM_NODES nodes..."
echo "---------------------------------------------"
echo "The following nodes have been allocated:"
scontrol show hostnames $SLURM_JOB_NODELIST
echo "---------------------------------------------"

# Running WITHOUT srun
python3 shakespeare.py 6
```

Submit it and examine the output:

```bash
sbatch shakespeare_broken.sbatch
```

In `answers.md`, answer:

   - How many different hostnames appear in the `scontrol show hostnames` output?
   - How many different hostnames appear in the sentence output?
   - Why is there a mismatch? Which node actually ran the Python script?


## Part C: Fix It with `srun`

The issue in Part B is that without `srun`, the command only executes on the **head node** of the allocation — the other 5 nodes sit idle. The `srun` command is what distributes execution across all allocated nodes.

Write a corrected batch script called `shakespeare_fixed.sbatch`:

```bash
#!/bin/bash
#SBATCH --job-name=shakespeare
#SBATCH --account=mpcs56430
#SBATCH --partition=caslake
#SBATCH --nodes=6
#SBATCH --ntasks-per-node=1
#SBATCH --time=00:05:00
#SBATCH --output=shakespeare-%j.out
#SBATCH --error=shakespeare-%j.err

echo "Writing our great work on $SLURM_JOB_NUM_NODES nodes..."
echo "---------------------------------------------"
echo "The following nodes have been allocated:"
scontrol show hostnames $SLURM_JOB_NODELIST
echo "---------------------------------------------"

# Running WITH srun — executes on every allocated node
srun python3 shakespeare.py 1
```

Submit it and examine the output.

In `answers.md`, answer:

   - How many different hostnames appear in the sentence output now?
   - Each node generated 1 sentence. How could you have each node generate a different number of sentences? (Hint: think about how `$SLURM_PROCID` could help.)


## Part D: Write a Book

Now let's put all 6 nodes to work writing a full "book." Each node will be responsible for writing one **chapter** — a batch of sentences that will be combined into a single output.

1. Update your script to accept a chapter number as an argument and write its output to a file:

```bash
python3 shakespeare.py --chapter 3 --sentences 20 --output chapters/
```

This should write 20 sentences to a file like `chapters/chapter_03.txt`.

2. Write a batch script called `write_book.sbatch` that uses `srun` so each of the 6 nodes writes its own chapter. Use `$SLURM_PROCID` (which is 0, 1, 2, ..., 5 — one per task) as the chapter number:

```bash
srun python3 shakespeare.py --chapter $SLURM_PROCID --sentences 20 --output chapters/
```

> **Note:** `$SLURM_PROCID` is a SLURM environment variable that is assigned within a multi-node job — each task gets a unique ID starting from 0. In this case, with 6 nodes, the values will be 0, 1, 2, 3, 4, and 5.

3. After the job completes, combine all chapters into a single file:

```bash
cat chapters/chapter_*.txt > the_complete_works.txt
```

4. In `answers.md`, answer:

   - How many total sentences are in your book?
   - Are the chapters interleaved or in order? Why?
   - What would happen if you requested `--nodes=12` instead of 6? How would that change your book?


## Part E: Automating the Pipeline with Job Dependencies

In Part D, you ran the `cat` command manually after the job finished to combine the chapters. But what if you wanted the entire pipeline — writing and combining — to run unattended, without you watching the queue?

SLURM supports **job dependencies**, which let you submit a second job that waits for a first job to finish before it starts. The `--dependency` flag tells SLURM the relationship between the two jobs.

1. Create a short batch script called `combine_book.sbatch` that combines the chapters:

```bash
#!/bin/bash
#SBATCH --job-name=combine_book
#SBATCH --account=mpcs56430
#SBATCH --partition=caslake
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=1G
#SBATCH --time=00:05:00
#SBATCH --output=combine-%j.out
#SBATCH --error=combine-%j.err

echo "Combining chapters into the complete works..."
cat chapters/chapter_*.txt > the_complete_works.txt
echo "Done. $(wc -l < the_complete_works.txt) lines written."
```

2. Now submit both jobs as a pipeline. The `--parsable` flag makes `sbatch` return just the job ID, which you can capture and pass to the second job:

```bash
# Submit the writing job and capture its job ID
JOB_ID=$(sbatch --parsable write_book.sbatch)
echo "Submitted writing job: $JOB_ID"

# Submit the combine job, which will wait until the writing job succeeds
sbatch --dependency=afterok:$JOB_ID combine_book.sbatch
```

The `afterok` dependency means the combine job will only start if the writing job completes **successfully** (exit code 0). If the writing job fails, the combine job will remain in the queue with a `DependencyNeverSatisfied` state. Other dependency types include `afterany` (run regardless of success or failure) and `afternotok` (run only if the first job fails).

3. Submit the pipeline and verify with `squeue` that the combine job shows a `(Dependency)` reason while the writing job runs.

4. In `answers.md`, explain: why is `afterok` a better choice than `afterany` for this pipeline?


## Deliverables

Your `problem_5` directory should contain:

```
problem_5/
├── answers.md
├── shakespeare.py
├── shakespeare_broken.sbatch
├── shakespeare_fixed.sbatch
├── write_book.sbatch
├── combine_book.sbatch
├── chapters/
└── the_complete_works.txt
```

**Submitting.** Put everything for this problem in `problem_5/` in your repository, with your results easy to find: your scripts, the output files your answers refer to, and `answers.md`. You can work in your clone on Midway3 and commit and push from there, or copy your work back to your laptop (for example with `scp`) and push from there. Either is fine.
